# IOAI — 2024 Summer National Transfer Learning (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('CIFAR-10/100 은 torchvision 으로, 사전학습 ResNet 는 torch.hub 로 자동 다운로드됩니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 전이학습 (Transfer Learning): CIFAR-100 → CIFAR-10 — 모범답안

HAIO 2024 여름 결선(CV). **CIFAR-100 사전학습 ResNet20**(torch.hub `cifar100_resnet20`)을 **CIFAR-10** 으로
전이한다. 점수 = **CIFAR-10 테스트 정확도**(표준 순서). 제출 `submission.csv`(id,label).

**전이학습 요령**: 사전학습 백본은 그대로 두고 **분류 헤드(fc)만 100→10 으로 교체**한 뒤 전체를 미세조정.
CIFAR-100 과 CIFAR-10 은 도메인이 매우 비슷해 사전학습 특징이 잘 옮겨간다. 이 모델은 **CIFAR 전용(32px)** 이라
업샘플이 필요 없다(ImageNet 백본과 대조). **SGD OneCycle·라벨스무딩** 로 25에폭 → 테스트 정확도 **≈0.92**
(헤드 교체 직후 미세조정 전엔 ~0.10, 스캐폴드의 3에폭 Adam 은 ~0.85).


In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader
torch.manual_seed(0); device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)

MEAN, STD = (0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)
tf_tr = T.Compose([T.RandomCrop(32,padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(MEAN,STD)])
tf_te = T.Compose([T.ToTensor(), T.Normalize(MEAN,STD)])
tr = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_tr)
te = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf_te)
dl_tr = DataLoader(tr, 256, shuffle=True,  num_workers=4, pin_memory=True)
dl_te = DataLoader(te, 512, shuffle=False, num_workers=4)
print("train", len(tr), "test", len(te))


In [ ]:
# CIFAR-100 사전학습 ResNet20 로드 + 헤드 100->10 교체 (전이학습)
model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar100_resnet20", pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)     # 새 10클래스 헤드
model = model.to(device)
print("params", sum(p.numel() for p in model.parameters()))

EPOCHS = 25
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4, nesterov=True)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=0.05, epochs=EPOCHS, steps_per_epoch=len(dl_tr))
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

@torch.no_grad()
def test_accuracy():
    model.eval(); c = t = 0
    for x, y in dl_te:
        p = model(x.to(device)).argmax(1).cpu(); c += (p==y).sum().item(); t += y.size(0)
    return c / t


In [ ]:
# 전체 미세조정 (SGD OneCycle + 라벨스무딩). ResNet20 은 작아 빠르다.
for ep in range(EPOCHS):
    model.train()
    for x, y in dl_tr:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); crit(model(x), y).backward(); opt.step(); sched.step()
    if (ep+1) % 5 == 0 or ep == EPOCHS-1:
        print(f"epoch {ep+1}/{EPOCHS}  test accuracy {test_accuracy():.4f}", flush=True)


In [ ]:
# 테스트 예측(표준 순서) -> submission.csv
model.eval(); preds = []
with torch.no_grad():
    for x, _ in dl_te: preds.append(model(x.to(device)).argmax(1).cpu().numpy())
preds = np.concatenate(preds)
pd.DataFrame({"id": range(len(preds)), "label": preds}).to_csv("submission.csv", index=False)
print("submission.csv 저장:", len(preds), "| 최종 테스트 정확도", round(test_accuracy(), 4))


### 정리
- CIFAR-100 사전학습 **ResNet20**(272K 파라미터) → 헤드 100→10 교체 → CIFAR-10 25에폭 미세조정 → 테스트 정확도 ≈ **0.92**.
- **핵심**: 소스(CIFAR-100)·타깃(CIFAR-10) 도메인이 유사해 전이가 잘 된다. 이 백본은 **CIFAR 전용 32px** 라
  업샘플 불필요(ImageNet 백본을 쓰는 문제와 대조 — 그건 64~224px 업샘플이 필요).
- **더 끌어올리려면**: 더 큰 백본(cifar100_resnet56 ≈0.94)·에폭↑·Mixup/CutMix·TTA.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)